# Ergebnis-Übersicht: Baseline vs. Finetuning-Runs

Dieses Notebook lädt Evaluierungs- und Trainingslogs aus `3_Model/runs/` und vergleicht sie gegen das DeepForest/Freudenberg-Baseline-Modell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

REPO = Path('/home/leafline/leafline')
RUNS = ['v1', 'v1_sampling_fix', 'v1_no_ndom']

RUN_COLORS = {
    'v1': '#2a78d6',              # blue
    'v1_sampling_fix': '#008300', # green
    'v1_no_ndom': '#e87ba4',      # magenta
    'baseline': '#eda100',        # yellow
}

## 1. Trainingsverlauf (pixelweises Val-F1)

In [ ]:
# Load training logs for each run
train_logs = {}
for run in RUNS:
    csv_path = REPO / '3_Model' / 'runs' / run / 'train_log.csv'
    if csv_path.exists():
        train_logs[run] = pd.read_csv(csv_path)
        print(f'{run}: loaded {len(train_logs[run])} epochs')
    else:
        print(f'{run}: train_log.csv NOT FOUND at {csv_path}')

# Plot val_f1 vs epoch
fig, ax = plt.subplots(figsize=(10, 6))

best_epochs_summary = []

for run in RUNS:
    if run in train_logs:
        df = train_logs[run]
        ax.plot(df['epoch'], df['val_f1'], 
                label=run, color=RUN_COLORS[run], linewidth=2)
        
        best_idx = df['val_f1'].idxmax()
        best_epoch = df.loc[best_idx, 'epoch']
        best_f1 = df.loc[best_idx, 'val_f1']
        
        ax.scatter([best_epoch], [best_f1], 
                   color=RUN_COLORS[run], s=100, zorder=5)
        
        best_epochs_summary.append({
            'Run': run,
            'Best Val-F1': f'{best_f1:.4f}',
            'Epoch': int(best_epoch)
        })

ax.set_xlabel('Epoche')
ax.set_ylabel('Val-F1 (pixelweise)')
ax.set_title('Trainingsverlauf pro Run')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary table
if best_epochs_summary:
    summary_df = pd.DataFrame(best_epochs_summary)
    print('\nBeste Val-F1 pro Run:')
    print(summary_df.to_string(index=False))
else:
    print('Keine Trainingslogs gefunden.')

## 2. Test-F1 (kronenweise, IoU-Matching) nach Run und Auflösung

In [ ]:
# Baseline data (from 2_BaselineModel/README.md) -- includes pred/gt/tp/fp/fn so it
# uses the exact same schema as eval_test.csv; this matters because the micro-average
# below sums tp/fp/fn across gebiet, and a baseline row without those columns would
# silently sum to 0 after concat (NaN.sum() == 0), zeroing out the baseline entirely.
baseline_data = {
    'gebiet': ['BotGarten', 'BotGarten', 'BotGarten', 'HoernNord', 'HoernNord', 'HoernNord'],
    'aufloesung': ['7.5cm', '20cm', '20cm-spring', '7.5cm', '20cm', '20cm-spring'],
    'pred': [16, 324, 83, 1, 139, 0],
    'gt': [357, 357, 357, 375, 375, 375],
    'tp': [0, 129, 18, 0, 74, 0],
    'fp': [16, 195, 65, 1, 65, 0],
    'fn': [357, 228, 339, 375, 301, 375],
    'precision': [0.000, 0.398, 0.217, 0.000, 0.532, 0.000],
    'recall': [0.000, 0.361, 0.051, 0.000, 0.197, 0.000],
    'f1': [0.000, 0.379, 0.082, 0.000, 0.288, 0.000]
}
baseline_df = pd.DataFrame(baseline_data)
baseline_df['run'] = 'baseline'

# Load eval_test.csv for each run
all_eval_data = []

for run in RUNS:
    csv_path = REPO / '3_Model' / 'runs' / run / 'eval_test.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df['run'] = run
        all_eval_data.append(df)
        print(f'{run}: loaded {len(df)} test results ({df["aufloesung"].unique().tolist()})')
    else:
        print(f'{run}: eval_test.csv NOT FOUND at {csv_path}')

# Combine all runs' eval data
if all_eval_data:
    eval_df = pd.concat(all_eval_data, ignore_index=True)
    eval_df = pd.concat([eval_df, baseline_df], ignore_index=True)
    print(f'\nCombined eval data: {len(eval_df)} rows')
    print(f'Resolutions available: {sorted(eval_df["aufloesung"].unique().tolist())}')
else:
    eval_df = baseline_df
    print('No finetuned eval data found; using baseline only.')

# Compute micro-averaged F1 per (run, aufloesung)
# Group by run and aufloesung, sum tp/fp/fn across gebiet, then compute metrics.
# All rows (baseline + finetuned) now share the same tp/fp/fn schema, so this is a
# real micro-average everywhere -- no NaN-fallback branch needed.

def compute_f1(tp, fp, fn):
    '''Compute F1 from confusion matrix, guard division by zero.'''
    if tp + fp == 0:
        precision = 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 0.0
    else:
        recall = tp / (tp + fn)
    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

summary_rows = []
for (run, aufloesung), group in eval_df.groupby(['run', 'aufloesung']):
    tp_sum = group['tp'].sum()
    fp_sum = group['fp'].sum()
    fn_sum = group['fn'].sum()
    prec, rec, f1 = compute_f1(tp_sum, fp_sum, fn_sum)

    summary_rows.append({
        'run': run,
        'aufloesung': aufloesung,
        'precision': prec,
        'recall': rec,
        'f1': f1
    })

summary_df = pd.DataFrame(summary_rows)
summary_pivot = summary_df.pivot(index='run', columns='aufloesung', values='f1')

print('\nMicro-averaged F1 by run and resolution:')
print(summary_pivot.round(4))

## 3. Vergleichsdiagramm: Baseline vs. Finetuning je Auflösung

In [ ]:
# Prepare data for grouped bar chart
resolutions = sorted(summary_df['aufloesung'].unique().tolist())
runs_with_data = sorted(summary_df['run'].unique().tolist())

fig, ax = plt.subplots(figsize=(12, 6))

# Create bars for each resolution group
bar_width = 0.18  # Width of each bar
group_spacing = 1.0  # Space between groups

# Map runs to positions within each group
run_positions = {run: i for i, run in enumerate(runs_with_data)}
num_runs = len(runs_with_data)

# Adjust bar positions to center the group
group_width = num_runs * bar_width + (num_runs - 1) * 0.02
group_offset = (group_width - bar_width) / 2

for res_idx, resolution in enumerate(resolutions):
    res_data = summary_df[summary_df['aufloesung'] == resolution]
    
    for run in runs_with_data:
        run_data = res_data[res_data['run'] == run]
        if len(run_data) > 0:
            f1_value = run_data.iloc[0]['f1']
            x_pos = res_idx * group_spacing + run_positions[run] * bar_width - group_offset
            ax.bar(x_pos, f1_value, width=bar_width, 
                   color=RUN_COLORS.get(run, '#cccccc'),
                   label=run if res_idx == 0 else '')

# Set x-axis ticks at resolution group centers
ax.set_xticks([i * group_spacing for i in range(len(resolutions))])
ax.set_xticklabels(resolutions)
ax.set_xlabel('Auflösung')
ax.set_ylabel('Micro-averaged F1 (kronenweise, IoU-matched)')
ax.set_title('F1 nach Auflösung: Baseline vs. Finetuning')
ax.set_ylim([0, 1.0])
ax.grid(True, alpha=0.3, axis='y')

# Create legend (remove duplicate labels)
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), loc='upper left')

plt.tight_layout()
plt.show()

print('Chart complete.')

## 4. Kurzfassung

In [ ]:
# Compute overall best run and improvement over baseline

baseline_f1_overall = baseline_df['f1'].mean()
finetuned_runs = [r for r in summary_df['run'].unique() if r != 'baseline']

if len(finetuned_runs) > 0:
    # Find best finetuned run
    best_f1_per_run = summary_df[summary_df['run'] != 'baseline'].groupby('run')['f1'].mean()
    best_run = best_f1_per_run.idxmax()
    best_f1 = best_f1_per_run.max()
    
    improvement = best_f1 - baseline_f1_overall
    improvement_pct = (improvement / baseline_f1_overall * 100) if baseline_f1_overall > 0 else 0
    
    print(f'Baseline micro-average F1: {baseline_f1_overall:.4f}')
    print(f'Best finetuned run: {best_run}')
    print(f'Best run micro-average F1: {best_f1:.4f}')
    print(f'Improvement: +{improvement:.4f} ({improvement_pct:+.1f}%)')
    print()
    print('Detailed breakdown by resolution:')
    for resolution in resolutions:
        res_summary = summary_df[summary_df['aufloesung'] == resolution]
        print(f'\n{resolution}:')
        for _, row in res_summary.iterrows():
            print(f'  {row["run"]}: F1={row["f1"]:.4f}')
else:
    print('Noch nicht genug Ergebnisse fuer eine vollstaendige Zusammenfassung.')
    print('Nur folgende Runs/Aufloesungen liegen vor:')
    print(summary_df.to_string())